<a href="https://colab.research.google.com/github/manugd1105/AI_challenge_UPF/blob/main/Primeras_pruebas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
# 2. Descomprimir el archivo LA (tarda ~ 3 min)
!unzip --q "/content/drive/MyDrive/Colab_Notebooks/Reto de Inteligencia Artificial/DS_10283_3336/LA.zip"

Se han truncado las últimas 5000 líneas del flujo de salida.
  inflating: LA/ASVspoof2019_LA_eval/flac/LA_E_7787040.flac  
  inflating: LA/ASVspoof2019_LA_eval/flac/LA_E_2924301.flac  
  inflating: LA/ASVspoof2019_LA_eval/flac/LA_E_9249366.flac  
  inflating: LA/ASVspoof2019_LA_eval/flac/LA_E_3442936.flac  
  inflating: LA/ASVspoof2019_LA_eval/flac/LA_E_7772915.flac  
  inflating: LA/ASVspoof2019_LA_eval/flac/LA_E_5569336.flac  
  inflating: LA/ASVspoof2019_LA_eval/flac/LA_E_7773607.flac  
  inflating: LA/ASVspoof2019_LA_eval/flac/LA_E_7813281.flac  
  inflating: LA/ASVspoof2019_LA_eval/flac/LA_E_9705954.flac  
  inflating: LA/ASVspoof2019_LA_eval/flac/LA_E_2427464.flac  
  inflating: LA/ASVspoof2019_LA_eval/flac/LA_E_1000273.flac  
  inflating: LA/ASVspoof2019_LA_eval/flac/LA_E_5263550.flac  
  inflating: LA/ASVspoof2019_LA_eval/flac/LA_E_1642109.flac  
  inflating: LA/ASVspoof2019_LA_eval/flac/LA_E_1339848.flac  
  inflating: LA/ASVspoof2019_LA_eval/flac/LA_E_9495857.flac  
  inflati

In [3]:
# 2. Descomprimir el archivo PA (tarda ~ 7 min)
## !unzip "/content/drive/MyDrive/Colab_Notebooks/Reto de Inteligencia Artificial/DS_10283_3336/PA.zip"

In [25]:
import os

def count_files_in_directory(path):
    if not os.path.exists(path):
        return f"La carpeta '{path}' no existe.", 0
    count = 0
    for root, dirs, files in os.walk(path):
        count += len(files)
    return count

# Paths for LA dataset splits
la_train_path = '/content/LA/ASVspoof2019_LA_train/flac'
la_dev_path = '/content/LA/ASVspoof2019_LA_dev/flac'
la_eval_path = '/content/LA/ASVspoof2019_LA_eval/flac'

# Count files for each split
num_files_la_train = count_files_in_directory(la_train_path)
num_files_la_dev = count_files_in_directory(la_dev_path)
num_files_la_eval = count_files_in_directory(la_eval_path)

print(f"Número de ficheros en la carpeta '{la_train_path}' (train): {num_files_la_train}")
print(f"Número de ficheros en la carpeta '{la_dev_path}' (dev): {num_files_la_dev}")
print(f"Número de ficheros en la carpeta '{la_eval_path}' (eval): {num_files_la_eval}")

Número de ficheros en la carpeta '/content/LA/ASVspoof2019_LA_train/flac' (train): 25380
Número de ficheros en la carpeta '/content/LA/ASVspoof2019_LA_dev/flac' (dev): 24986
Número de ficheros en la carpeta '/content/LA/ASVspoof2019_LA_eval/flac' (eval): 71933


## Conversión a datos numéricos

In [34]:
# Instalamos librosa para procesamiento de audio
!pip install librosa soundfile

import os
import librosa
import numpy as np

# Cuantos registros quiero coger para mi dataframe (25380 el total en train)
# Para la carpeta 'eval' hay 71933 ficheros, vamos a coger todos
num_registros = 71933

# Rutas al dataset LA
# Cambiamos las rutas para usar el protocolo y los archivos de audio de evaluación.
la_protocol_path = '/content/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt'
# Usamos la ruta a los archivos de audio de evaluación.
la_audio_path = '/content/LA/ASVspoof2019_LA_eval/flac'

# Función para extraer características de un archivo de audio
def extract_features(audio_file_path, sr=16000, n_mfcc=13):
    try:
        # Cargar el archivo de audio
        audio, sample_rate = librosa.load(audio_file_path, sr=sr)
        # Extraer MFCCs
        mfccs = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=n_mfcc)
        # Transponer para tener el formato (n_mfcc, time_steps)
        return mfccs.T
    except Exception as e:
        print(f"Error procesando {audio_file_path}: {e}")
        return None

# Leemos el archivo de protocolo para obtener una lista de audios y sus etiquetas
sample_audio_files = []
labels = []

with open(la_protocol_path, 'r') as f:
    for i, line in enumerate(f):
        # Ejemplo de línea del protocolo eval: LA_0000 LA_E_2628468 - A06 spoof
        parts = line.strip().split(' ')
        if len(parts) >= 3:
            # El ID del archivo de audio es el segundo elemento en el protocolo del ASVspoof2019 eval.
            file_id = parts[1]
            label = parts[-1]
            full_audio_path = os.path.join(la_audio_path, f'{file_id}.flac')
            if os.path.exists(full_audio_path):
                sample_audio_files.append(full_audio_path)
                labels.append(label)
        if i >= num_registros - 1: # Leer hasta num_registros
            break

if not sample_audio_files:
    print(f"No se encontraron archivos de audio en la ruta: {la_audio_path} o el protocolo está vacío/malformado.")
else:
    print(f"Procesando {len(sample_audio_files)} archivos de audio de ejemplo...")
    # Convertir etiquetas a formato numérico
    # 'bonafide' -> 0, 'spoof' -> 1
    label_mapping = {'bonafide': 0, 'spoof': 1}
    numerical_labels = [label_mapping.get(lbl, -1) for lbl in labels]

    # Extraer características para los archivos de ejemplo
    all_features = []
    for i, audio_file in enumerate(sample_audio_files):
        features = extract_features(audio_file)
        if features is not None:
            all_features.append(features)
            print(f"Archivo: {os.path.basename(audio_file)}, MFCCs shape: {features.shape}, Etiqueta numérica: {numerical_labels[i]}")

    if all_features:
        print("\nCaracterísticas extraídas para los archivos. Cada elemento en 'all_features' es un array numpy de MFCCs para un archivo de audio.")
        print("Puedes apilarlos o rellenarlos para que tengan la misma longitud si vas a entrenar un modelo que requiere entradas de tamaño fijo.")
    else:
        print("No se pudieron extraer características de ningón archivo de audio.")

Se han truncado las últimas 5000 líneas del flujo de salida.
Archivo: LA_E_3263504.flac, MFCCs shape: (95, 13), Etiqueta numérica: 1
Archivo: LA_E_1402939.flac, MFCCs shape: (58, 13), Etiqueta numérica: 1
Archivo: LA_E_2178031.flac, MFCCs shape: (120, 13), Etiqueta numérica: 1
Archivo: LA_E_2747279.flac, MFCCs shape: (179, 13), Etiqueta numérica: 1
Archivo: LA_E_1069211.flac, MFCCs shape: (101, 13), Etiqueta numérica: 1
Archivo: LA_E_7954006.flac, MFCCs shape: (41, 13), Etiqueta numérica: 1
Archivo: LA_E_5994036.flac, MFCCs shape: (54, 13), Etiqueta numérica: 1
Archivo: LA_E_9252990.flac, MFCCs shape: (75, 13), Etiqueta numérica: 0
Archivo: LA_E_1963724.flac, MFCCs shape: (110, 13), Etiqueta numérica: 1
Archivo: LA_E_9703165.flac, MFCCs shape: (104, 13), Etiqueta numérica: 1
Archivo: LA_E_4326729.flac, MFCCs shape: (36, 13), Etiqueta numérica: 1
Archivo: LA_E_4769510.flac, MFCCs shape: (103, 13), Etiqueta numérica: 0
Archivo: LA_E_2025771.flac, MFCCs shape: (54, 13), Etiqueta numérica:

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_19843/1257356599.py", line 62, in <cell line: 0>
    features = extract_features(audio_file)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_19843/1257356599.py", line 24, in extract_features
    mfccs = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=n_mfcc)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/librosa/feature/spectral.py", line 1993, in mfcc
    S = power_to_db(melspectrogram(y=y, sr=sr, norm = mel_norm, **kwargs))
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/librosa/feature/spectral.py", line 2148, in melspectrogram
    mel_basis = filters.mel(sr=sr, n_fft=n_fft, **kwargs)
              

TypeError: object of type 'NoneType' has no len()

Vamos a convertirlo a un dataframe de pandas

In [29]:
import pandas as pd
import os
import numpy as np # Import numpy for statistical operations
import scipy.stats # Import scipy.stats for skewness and kurtosis

# To retrieve the SPEAKER_ID and SYSTEM_ID, we need to re-read the protocol file
# and match the audio file names. la_protocol_path is available from the kernel state.

speaker_ids = []
system_ids = []

# Create a mapping from audio_file_name (e.g., 'LA_D_1047731') to its protocol info
protocol_map = {}
with open(la_protocol_path, 'r') as f:
    for line in f:
        parts = line.strip().split(' ')
        if len(parts) >= 5: # Ensure all expected parts are present
            speaker_id = parts[0]
            audio_file_name = parts[1]
            system_id = parts[3] # SYSTEM_ID is the 4th element (index 3)
            # The 5th element (index 4) is the KEY, which corresponds to our 'labels' list
            protocol_map[audio_file_name] = {'speaker_id': speaker_id, 'system_id': system_id}

# Crear una lista de diccionarios para cada entrada
data = []
for i in range(len(sample_audio_files)):
    file_name_with_ext = os.path.basename(sample_audio_files[i])
    file_id_without_ext = os.path.splitext(file_name_with_ext)[0] # e.g., 'LA_D_1047731'

    speaker_id = None
    system_id = None
    if file_id_without_ext in protocol_map:
        info = protocol_map[file_id_without_ext]
        speaker_id = info['speaker_id']
        system_id = info['system_id']

    # Get mfcc_features for the current audio file
    current_mfcc_features = all_features[i]

    # Calculate statistical summaries (mean and std) for each MFCC coefficient
    # over the temporal windows (axis=0)
    mfcc_means = np.mean(current_mfcc_features, axis=0)
    mfcc_stds = np.std(current_mfcc_features, axis=0)
    mfcc_medians = np.median(current_mfcc_features, axis=0)
    mfcc_mins = np.min(current_mfcc_features, axis=0)
    mfcc_maxs = np.max(current_mfcc_features, axis=0)
    mfcc_skewness = scipy.stats.skew(current_mfcc_features, axis=0)
    mfcc_kurtosis = scipy.stats.kurtosis(current_mfcc_features, axis=0)

    entry = {
        'audio_file': sample_audio_files[i],
        'speaker_id': speaker_id,
        'system_id': system_id,
        'label_numerical': numerical_labels[i], # Renaming 'label' to 'label_numerical' for clarity
        'label_text': labels[i], # Add the original text label for clarity (bonafide/spoof)
        'num_temporal_windows': current_mfcc_features.shape[0] # Keep this for context
    }

    # Add mean MFCCs as individual columns
    for j in range(len(mfcc_means)):
        entry[f'mfcc_mean_{j}'] = mfcc_means[j]

    # Add standard deviation MFCCs as individual columns
    for j in range(len(mfcc_stds)):
        entry[f'mfcc_std_{j}'] = mfcc_stds[j]

    # Add median MFCCs as individual columns
    for j in range(len(mfcc_medians)):
        entry[f'mfcc_median_{j}'] = mfcc_medians[j]

    # Add min MFCCs as individual columns
    for j in range(len(mfcc_mins)):
        entry[f'mfcc_min_{j}'] = mfcc_mins[j]

    # Add max MFCCs as individual columns
    for j in range(len(mfcc_maxs)):
        entry[f'mfcc_max_{j}'] = mfcc_maxs[j]

    # Add skewness MFCCs as individual columns
    for j in range(len(mfcc_skewness)):
        entry[f'mfcc_skew_{j}'] = mfcc_skewness[j]

    # Add kurtosis MFCCs as individual columns
    for j in range(len(mfcc_kurtosis)):
        entry[f'mfcc_kurt_{j}'] = mfcc_kurtosis[j]

    data.append(entry)

# Convertir la lista de diccionarios en un DataFrame de pandas
df_features = pd.DataFrame(data)

print("DataFrame de características creado:")
display(df_features.head())
print(f"\nDimensiones del DataFrame: {df_features.shape}")
print(f"Columnas del DataFrame: {df_features.columns.tolist()}")

DataFrame de características creado:


,audio_file,speaker_id,system_id,label_numerical,label_text,num_temporal_windows,mfcc_mean_0,mfcc_mean_1,mfcc_mean_2,mfcc_mean_3,...,mfcc_kurt_3,mfcc_kurt_4,mfcc_kurt_5,mfcc_kurt_6,mfcc_kurt_7,mfcc_kurt_8,mfcc_kurt_9,mfcc_kurt_10,mfcc_kurt_11,mfcc_kurt_12
0,/content/LA/ASVspoof2019_LA_train/flac/LA_T_11...,LA_0079,-,0,bonafide,109,-310.407959,43.036842,-18.052126,8.644586,...,1.309874,-0.983724,0.045031,-0.580553,-0.504889,-0.063759,-0.427871,-0.748618,-0.302753,1.553051
1,/content/LA/ASVspoof2019_LA_train/flac/LA_T_12...,LA_0079,-,0,bonafide,138,-330.957123,50.894707,-19.692842,10.769039,...,1.119556,-0.531304,-0.649178,-1.053209,-0.370049,-0.978057,-0.795056,0.493778,1.114291,-0.034808
2,/content/LA/ASVspoof2019_LA_train/flac/LA_T_12...,LA_0079,-,0,bonafide,91,-352.316956,18.402121,0.244028,8.822980,...,0.983940,0.011368,0.288825,0.926077,0.828593,-0.113401,-0.939718,-0.463290,-0.476926,-0.569304
3,/content/LA/ASVspoof2019_LA_train/flac/LA_T_12...,LA_0079,-,0,bonafide,88,-280.845978,48.768970,-7.991325,21.069437,...,-0.633393,-0.965992,-0.608938,0.734988,-0.698169,-0.910887,-0.984191,-0.787603,-0.192135,-1.100056
4,/content/LA/ASVspoof2019_LA_train/flac/LA_T_13...,LA_0079,-,0,bonafide,111,-301.708405,63.264992,-9.561450,-3.289454,...,0.150839,0.560195,-0.117506,-0.600111,-0.150093,-1.282657,-1.159197,-0.441457,-0.084253,0.779144



Dimensiones del DataFrame: (25380, 97)
Columnas del DataFrame: ['audio_file', 'speaker_id', 'system_id', 'label_numerical', 'label_text', 'num_temporal_windows', 'mfcc_mean_0', 'mfcc_mean_1', 'mfcc_mean_2', 'mfcc_mean_3', 'mfcc_mean_4', 'mfcc_mean_5', 'mfcc_mean_6', 'mfcc_mean_7', 'mfcc_mean_8', 'mfcc_mean_9', 'mfcc_mean_10', 'mfcc_mean_11', 'mfcc_mean_12', 'mfcc_std_0', 'mfcc_std_1', 'mfcc_std_2', 'mfcc_std_3', 'mfcc_std_4', 'mfcc_std_5', 'mfcc_std_6', 'mfcc_std_7', 'mfcc_std_8', 'mfcc_std_9', 'mfcc_std_10', 'mfcc_std_11', 'mfcc_std_12', 'mfcc_median_0', 'mfcc_median_1', 'mfcc_median_2', 'mfcc_median_3', 'mfcc_median_4', 'mfcc_median_5', 'mfcc_median_6', 'mfcc_median_7', 'mfcc_median_8', 'mfcc_median_9', 'mfcc_median_10', 'mfcc_median_11', 'mfcc_median_12', 'mfcc_min_0', 'mfcc_min_1', 'mfcc_min_2', 'mfcc_min_3', 'mfcc_min_4', 'mfcc_min_5', 'mfcc_min_6', 'mfcc_min_7', 'mfcc_min_8', 'mfcc_min_9', 'mfcc_min_10', 'mfcc_min_11', 'mfcc_min_12', 'mfcc_max_0', 'mfcc_max_1', 'mfcc_max_2', 'mf

Tenemos las columnas:

- speaker_id: ID del hablante, con formato LA_****. Es quien ha hablado o a quien se está intentando imitar con el deepfake.

- system_id: ID of the speech spoofing system (A01 - A19),  or, for bonafide speech SYSTEM-ID is left blank ('-')

- label: 0 es bonafide (genuino) y 1 es spoof (fake).

In [30]:
df_features

,audio_file,speaker_id,system_id,label_numerical,label_text,num_temporal_windows,mfcc_mean_0,mfcc_mean_1,mfcc_mean_2,mfcc_mean_3,...,mfcc_kurt_3,mfcc_kurt_4,mfcc_kurt_5,mfcc_kurt_6,mfcc_kurt_7,mfcc_kurt_8,mfcc_kurt_9,mfcc_kurt_10,mfcc_kurt_11,mfcc_kurt_12
0,/content/LA/ASVspoof2019_LA_train/flac/LA_T_11...,LA_0079,-,0,bonafide,109,-310.407959,43.036842,-18.052126,8.644586,...,1.309874,-0.983724,0.045031,-0.580553,-0.504889,-0.063759,-0.427871,-0.748618,-0.302753,1.553051
1,/content/LA/ASVspoof2019_LA_train/flac/LA_T_12...,LA_0079,-,0,bonafide,138,-330.957123,50.894707,-19.692842,10.769039,...,1.119556,-0.531304,-0.649178,-1.053209,-0.370049,-0.978057,-0.795056,0.493778,1.114291,-0.034808
2,/content/LA/ASVspoof2019_LA_train/flac/LA_T_12...,LA_0079,-,0,bonafide,91,-352.316956,18.402121,0.244028,8.822980,...,0.983940,0.011368,0.288825,0.926077,0.828593,-0.113401,-0.939718,-0.463290,-0.476926,-0.569304
3,/content/LA/ASVspoof2019_LA_train/flac/LA_T_12...,LA_0079,-,0,bonafide,88,-280.845978,48.768970,-7.991325,21.069437,...,-0.633393,-0.965992,-0.608938,0.734988,-0.698169,-0.910887,-0.984191,-0.787603,-0.192135,-1.100056
4,/content/LA/ASVspoof2019_LA_train/flac/LA_T_13...,LA_0079,-,0,bonafide,111,-301.708405,63.264992,-9.561450,-3.289454,...,0.150839,0.560195,-0.117506,-0.600111,-0.150093,-1.282657,-1.159197,-0.441457,-0.084253,0.779144
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25375,/content/LA/ASVspoof2019_LA_train/flac/LA_T_97...,LA_0098,A06,1,spoof,127,-265.135193,69.970108,-4.350155,18.199944,...,0.262429,-1.014055,-0.671720,-1.147501,-0.231507,-0.934747,-0.392647,-0.509243,0.701681,0.266892
25376,/content/LA/ASVspoof2019_LA_train/flac/LA_T_97...,LA_0098,A06,1,spoof,72,-263.532379,88.049995,-5.463266,23.480251,...,0.003089,-1.116645,-1.350957,-1.368827,-0.596425,-0.749630,-0.015985,-0.677207,0.515662,-0.996413
25377,/content/LA/ASVspoof2019_LA_train/flac/LA_T_97...,LA_0098,A06,1,spoof,129,-306.537842,68.004944,-3.615197,18.910183,...,0.665376,-0.861691,-0.820491,-1.343166,-0.572601,0.668659,-0.506145,0.737168,0.987404,-0.989994
25378,/content/LA/ASVspoof2019_LA_train/flac/LA_T_98...,LA_0098,A06,1,spoof,98,-297.404694,59.802021,12.725373,30.625031,...,0.200634,-0.944447,-0.687091,-0.981270,-1.225270,-0.893088,-0.452085,-1.116231,0.583324,-1.328375


In [31]:
df_features.system_id.unique()

array(['-', 'A01', 'A02', 'A03', 'A04', 'A05', 'A06'], dtype=object)

In [32]:
df_features.to_csv('df_train_manu.csv', index=False)